In [13]:
!pip install opencv-python pillow


In [9]:
pip install --upgrade pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 89.2 MB/s eta 0:00:00


In [15]:
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import csv

# =====================================================
# Configuration
# =====================================================
# Folder where your extracted images are stored
IMG_DIR = Path("/content/figures_extracted")

# Output CSV file with scores and classification
OUT_CSV = Path("/content/figures_scored.csv")

# Image extensions to consider
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def compute_image_metrics(img_path: Path):
    """
    Compute simple metrics to help distinguish diagrams from photos:
      - white_ratio: fraction of bright pixels (close to white)
      - edge_density: fraction of edge pixels from Canny
      - color_std: standard deviation of colors (higher for photos)
    """
    # Load image with Pillow, then convert to numpy array
    pil_img = Image.open(img_path).convert("RGB")
    img = np.array(pil_img)

    # Convert to grayscale for edge + brightness analysis
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Normalize grayscale to [0,1]
    gray_norm = gray.astype(np.float32) / 255.0

    # 1) White ratio: how much of the image is near white
    #    Diagrams usually have large white background.
    white_threshold = 0.85  # pixel considered "white" if brightness > 0.85
    white_ratio = float((gray_norm > white_threshold).mean())

    # 2) Edge density: run Canny and compute ratio of edge pixels
    edges = cv2.Canny((gray_norm * 255).astype(np.uint8), 100, 200)
    edge_density = float((edges > 0).mean())

    # 3) Color standard deviation: photos usually have higher color variation
    #    Flatten over spatial dimensions, compute std over all RGB values.
    color_std = float(img.reshape(-1, 3).astype(np.float32).std())

    return white_ratio, edge_density, color_std


def classify_diagram(white_ratio, edge_density, color_std):
    """
    Heuristic classification:
      - diagrams typically:
        * have significant white background
        * have noticeable edges
        * have lower color variance than photos
    Thresholds are empirical and may be fine-tuned.
    """
    # Basic heuristic thresholds (tune if needed)
    min_white_ratio = 0.35     # at least 35% of pixels are bright
    min_edge_density = 0.005   # some edges present
    max_color_std = 80.0       # not too "colorful" or textured

    if (white_ratio >= min_white_ratio and
        edge_density >= min_edge_density and
        color_std <= max_color_std):
        return 1  # diagram-like
    else:
        return 0  # not diagram-like


def crawl_and_score_images(img_dir: Path, out_csv: Path):
    """
    Crawl all images in img_dir, compute metrics, classify as diagram/non-diagram,
    and write results into CSV.
    """
    if not img_dir.exists():
        print(f"[ERROR] Image directory not found: {img_dir}")
        return

    image_files = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    image_files.sort()

    if not image_files:
        print(f"[WARN] No image files found in {img_dir}")
        return

    print(f"[INFO] Found {len(image_files)} images in {img_dir}")

    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        # CSV header
        writer.writerow([
            "filename",
            "white_ratio",
            "edge_density",
            "color_std",
            "is_diagram"
        ])

        for img_path in image_files:
            try:
                white_ratio, edge_density, color_std = compute_image_metrics(img_path)
                is_diagram = classify_diagram(white_ratio, edge_density, color_std)

                writer.writerow([
                    img_path.name,
                    f"{white_ratio:.4f}",
                    f"{edge_density:.4f}",
                    f"{color_std:.2f}",
                    int(is_diagram),
                ])

                print(f"[OK] {img_path.name} -> "
                      f"white={white_ratio:.3f}, edges={edge_density:.3f}, "
                      f"std={color_std:.1f}, diagram={is_diagram}")
            except Exception as e:
                print(f"[ERROR] Failed on {img_path.name}: {e}")

    print(f"\n[INFO] Scores saved to: {out_csv}")


# Run the crawler
crawl_and_score_images(IMG_DIR, OUT_CSV)


[WARN] No image files found in /content/figures_extracted


In [12]:
import os

os.listdir("/content")


['.config',
 'Boylestad_Electronic_Devices_and_Circuit_Theory_11th.pdf',
 '.ipynb_checkpoints',
 'figures_extracted',
 'sample_data']